# Ellipsometry 互動擬合 Notebook

**用途：** 探索資料、試不同模型、調參數、看結果。

**對應 GUI：** 這個 notebook 涵蓋 GUI 所有功能，但更彈性（可寫程式邏輯）。

**內容：**
1. 載入資料 + 預覽
2. 偽介電函數探索（決定要幾個振盪器）
3. 簡單 fit：thickness 用內建材料
4. 進階 fit：GenOsc 自組振盪器
5. 三種模式比較（both / e2_only / e1_only）
6. 不確定度分析（lmfit emcee MCMC）

## 0. 環境設定

In [ ]:
import sys, os
from pathlib import Path

# 把專案根目錄加到 path（讓 ellipsometry 套件可被 import）
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ellipsometry.core.io import read_reflection, read_transmission
from ellipsometry.core.dispersion import MaterialLibrary, create_dispersion
from ellipsometry.core.tmm_calc import Layer, calculate, pseudo_epsilon
from ellipsometry.core.fitter import Fitter

print('內建材料庫:', MaterialLibrary.list_available())

## 1. 載入資料

用內建範例：50 nm Au / sapphire。

In [ ]:
SAMPLE = 'sample1_Au50_sapphire'   # 改這裡換樣品：sample1_Au50_sapphire / sample2_SiO2_285_Si / sample3_bare_Si

data = read_reflection(ROOT / f'data/{SAMPLE}/R_long.dat')
T_data = read_transmission(ROOT / f'data/{SAMPLE}/T.dat')
print(data)
print(T_data)

In [ ]:
# 預覽：Ψ, Δ vs λ
fig = make_subplots(rows=1, cols=2, subplot_titles=('Ψ (deg)', 'Δ (deg)'))
for j, ang in enumerate(data.angles):
    fig.add_trace(go.Scatter(x=data.wavelength, y=data.psi[:, j],
                              name=f'AOI={ang}°', mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.wavelength, y=data.delta[:, j],
                              name=f'AOI={ang}°', showlegend=False, mode='lines'), row=1, col=2)
fig.update_xaxes(title_text='Wavelength (nm)')
fig.update_layout(height=400, hovermode='x unified')
fig.show()

## 2. 偽介電函數 (pseudo-ε) 探索

用 ε2 的峰位置決定要放幾個 Lorentz 振盪器、各自能量在哪。

**注意：** pseudo-ε 假設樣品是半無限均勻體，對「薄膜+基板」的結果是「視在 ε」不是真實薄膜 ε，但**形狀和峰位置可信**。

In [ ]:
from ellipsometry.core.units import nm_to_eV

fig = make_subplots(rows=1, cols=2, subplot_titles=('<ε₁>', '<ε₂>'))
for j, ang in enumerate(data.angles):
    eps = pseudo_epsilon(data.psi[:, j], data.delta[:, j], ang)
    fig.add_trace(go.Scatter(x=nm_to_eV(data.wavelength), y=eps.real,
                              name=f'AOI={ang}°', mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=nm_to_eV(data.wavelength), y=eps.imag,
                              name=f'AOI={ang}°', showlegend=False, mode='lines'), row=1, col=2)
fig.update_xaxes(title_text='Energy (eV)')
fig.update_layout(height=400, hovermode='x unified')
fig.show()

print('觀察 <ε₂> 的峰位置 → 振盪器中心能量 (En)')
print('觀察 <ε₂> 峰寬 → 振盪器寬度 (Br)')
print('觀察 <ε₂> 峰高 → 振盪器振幅 (Amp)')

## 3. 簡單 fit：只 fit 厚度（用內建材料）

已知材料（如已知是 Au）→ 直接用內建 nk 表，只 fit 厚度。

In [ ]:
config_simple = {
    'layers': [
        {'name': 'air',  'material': 'air', 'thickness': 'infinite'},
        {'name': 'film', 'material': 'Au',     # 內建材料
         'thickness': 80,                       # 初始厚度（故意設錯，真值 50）
         'coherent': True,
         'fit_thickness': True,
         'thickness_bounds': [10, 200]},
        {'name': 'substrate', 'material': 'Al2O3',
         'thickness': 1_000_000, 'coherent': False, 'fit_thickness': False},
    ],
    'fit': {
        'target': 'both',
        'method': 'leastsq',
        'max_iter': 35,
        'delta_residual_mode': 'sin_cos',
        'loss': 'linear',
        'weighting': {'sigma_psi': 0.5, 'sigma_delta': 1.0, 'sigma_T': 0.01},
    },
}

fitter = Fitter(config_simple, data, T_data)
result = fitter.fit(verbose=True)

In [ ]:
# 對比圖：量測 vs 擬合
def plot_fit_comparison(result):
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=('Ψ (deg)', 'Δ (deg)', 'T', 'n, k 色散'))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for j, ang in enumerate(result.angles):
        c = colors[j % len(colors)]
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.psi_meas[:, j],
                                  name=f'量測 {ang}°', mode='markers',
                                  marker=dict(size=3, color=c)), row=1, col=1)
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.psi_fit[:, j],
                                  name=f'擬合 {ang}°', mode='lines',
                                  line=dict(color=c)), row=1, col=1)
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.delta_meas[:, j],
                                  mode='markers', marker=dict(size=3, color=c),
                                  showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.delta_fit[:, j],
                                  mode='lines', line=dict(color=c),
                                  showlegend=False), row=1, col=2)
    if result.T_meas is not None:
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.T_meas,
                                  mode='markers', marker=dict(size=3), name='量測 T'), row=2, col=1)
        fig.add_trace(go.Scatter(x=result.wavelength, y=result.T_fit,
                                  mode='lines', name='擬合 T'), row=2, col=1)
    # n, k
    n, k = result.layers[1].material.n_k(result.wavelength)
    fig.add_trace(go.Scatter(x=result.wavelength, y=n, name='n', line=dict(width=2)), row=2, col=2)
    fig.add_trace(go.Scatter(x=result.wavelength, y=k, name='k', line=dict(width=2, dash='dash')),
                  row=2, col=2)
    fig.update_xaxes(title_text='Wavelength (nm)')
    fig.update_layout(height=700, hovermode='x unified',
                      title=f'MSE = {result.mse:.4e}, {result.n_iter} evals')
    return fig

plot_fit_comparison(result).show()

## 4. 進階 fit：GenOsc（自組振盪器）

適用於「不知道材料是什麼」或「想要 KK 一致的色散模型」。

從 pseudo-ε 觀察到的峰位置設振盪器初始值，然後讓擬合精調。

In [ ]:
config_genosc = {
    'layers': [
        {'name': 'air', 'material': 'air', 'thickness': 'infinite'},
        {'name': 'film',
         'material': {
             'model': 'gen_osc',
             'layer': {
                 'e1_offset': 1.0,
                 'poles': {
                     'uv': {'position': 20.0, 'magnitude': 100.0},
                 },
             },
             'oscillators': [
                 {'type': 'lorentz', 'amp': 3.0, 'en': 2.5, 'br': 0.3, 'active': True},
                 {'type': 'lorentz', 'amp': 2.0, 'en': 4.0, 'br': 0.5, 'active': True},
             ],
             'params': {                     # 同上，給 fitter 找 path
                 'e1_offset': 1.0,
                 'oscillators': [
                     {'amp': 3.0, 'en': 2.5, 'br': 0.3},
                     {'amp': 2.0, 'en': 4.0, 'br': 0.5},
                 ],
             },
             'fit': [
                 'e1_offset',
                 'oscillators[*].amp',
                 'oscillators[*].en',
                 'oscillators[*].br',
             ],
             'bounds': {
                 'e1_offset': [0.5, 5.0],
                 'oscillators[*].amp': [0, 10],
                 'oscillators[*].en': [0.5, 6.5],
                 'oscillators[*].br': [0.05, 2.0],
             },
         },
         'thickness': 50,
         'coherent': True,
         'fit_thickness': True,
         'thickness_bounds': [10, 200],
        },
        {'name': 'substrate', 'material': 'Al2O3',
         'thickness': 1_000_000, 'coherent': False, 'fit_thickness': False},
    ],
    'fit': {
        'target': 'both',
        'method': 'leastsq',
        'max_iter': 50,
        'delta_residual_mode': 'sin_cos',
        'loss': 'soft_l1',                    # 對 outlier 較穩
        'weighting': {'sigma_psi': 0.5, 'sigma_delta': 1.0, 'sigma_T': 0.01},
    },
}

fitter_go = Fitter(config_genosc, data, T_data)
result_go = fitter_go.fit(verbose=True)

In [ ]:
plot_fit_comparison(result_go).show()

## 5. 比較三種 fit 模式

**注意：** e2_only / e1_only 需要先用 both 鎖定厚度。下面把厚度固定後比較三模式。

In [ ]:
import copy
results_by_target = {}
for tgt in ['both']:    # e2/e1 因為 GenOsc 反推較慢，notebook 預設只跑 both
    cfg = copy.deepcopy(config_genosc)
    cfg['fit']['target'] = tgt
    if tgt != 'both':
        cfg['layers'][1]['fit_thickness'] = False    # e2/e1 不能 fit 厚度
    r = Fitter(cfg, data, T_data).fit(verbose=False)
    results_by_target[tgt] = r
    print(f'  {tgt}: MSE = {r.mse:.4e}, evals = {r.n_iter}')

# 要跑 e2/e1 取消下面註解（會慢很多，因為要 point-by-point 反演每個 AOI）
# for tgt in ['e2_only', 'e1_only']:
#     cfg = copy.deepcopy(config_genosc)
#     cfg['fit']['target'] = tgt
#     cfg['layers'][1]['fit_thickness'] = False
#     r = Fitter(cfg, data, T_data).fit(verbose=False)
#     results_by_target[tgt] = r
#     print(f'  {tgt}: MSE = {r.mse:.4e}, evals = {r.n_iter}')

## 6. 不確定度分析（lmfit MCMC，可選）

對最終擬合結果跑 emcee 取樣，得到參數的真實 posterior。

**警告：** 每個 sample 約 0.5-2 秒，1000 個 sample = 10-30 分鐘。

In [ ]:
# 取消下面 raise 來啟用
raise NotImplementedError('預設跳過 MCMC（很慢）。要跑請刪掉這行。')

import lmfit
mini = lmfit.Minimizer(fitter._residual_both, result.params._lmfit_result.params)
emcee_result = mini.emcee(
    nwalkers=20, steps=200, burn=50, thin=2, workers=1,
    progress=True,
)

# 印每個參數的中位數 + 16/84 percentile
for p in emcee_result.flatchain.columns:
    vals = emcee_result.flatchain[p]
    print(f'{p}: {vals.median():.4g} +{vals.quantile(0.84)-vals.median():.3g} '
          f'-{vals.median()-vals.quantile(0.16):.3g}')

## 7. 匯出結果

輸出 nk 表 + 擬合曲線 csv，與 config yaml。

In [ ]:
import yaml
OUT = ROOT / 'results' / SAMPLE
OUT.mkdir(parents=True, exist_ok=True)

# 1. nk 表
n, k = result.layers[1].material.n_k(result.wavelength)
pd.DataFrame({'wavelength_nm': result.wavelength, 'n': n, 'k': k}).to_csv(
    OUT / 'fitted_nk.csv', index=False)

# 2. 擬合曲線
rows = []
for j, a in enumerate(result.angles):
    for i, wl in enumerate(result.wavelength):
        rows.append({
            'wavelength_nm': wl, 'aoi_deg': a,
            'psi_meas': result.psi_meas[i, j], 'psi_fit': result.psi_fit[i, j],
            'delta_meas': result.delta_meas[i, j], 'delta_fit': result.delta_fit[i, j],
        })
pd.DataFrame(rows).to_csv(OUT / 'fit_curves.csv', index=False)

# 3. config
with open(OUT / 'config.yaml', 'w') as f:
    yaml.dump(config_simple, f, allow_unicode=True, default_flow_style=False)

print(f'✓ 已寫入 {OUT}')
print(f'  - fitted_nk.csv')
print(f'  - fit_curves.csv')
print(f'  - config.yaml')